In [ ]:
from sklearn.utils import resample
import os

fixed_part = X_train.iloc[:, :51].copy()  # ستون 0 تا 50
variable_part = X_train.iloc[:, 51:320].copy()  # ستون 51 تا 319 (268 ستون)
cluster_col = X_train.iloc[:, 320].copy()  # ستون cluster (عدد 320 ام)

clusters = cluster_col
cluster_counts = clusters.value_counts()
max_cluster_size = cluster_counts.max()

# حذف فایل‌های قبلی و ذخیره داده اصلی با ستون cluster هم اضافه شده
import os
if os.path.exists('X_train_augmented.csv'):
    os.remove('X_train_augmented.csv')
if os.path.exists('y_train_augmented.csv'):
    os.remove('y_train_augmented.csv')

# داده اصلی با ستون cluster
X_train_base = pd.concat([fixed_part.reset_index(drop=True),
                          variable_part.reset_index(drop=True),
                          cluster_col.reset_index(drop=True)], axis=1)
X_train_base.to_csv('X_train_augmented.csv', index=False)
pd.DataFrame(y_train, columns=['CCS']).to_csv('y_train_augmented.csv', index=False)

print("داده اصلی ذخیره شد.")

# افزوده سازی داده‌ها
for cluster_id, count in cluster_counts.items():
    if count < max_cluster_size:
        n_to_generate = max_cluster_size - count
        mask = clusters == cluster_id

        fixed_subset = fixed_part[mask]
        variable_subset = variable_part[mask]
        cluster_subset = cluster_col[mask]  # این هم باید تکرار بشه
        y_subset = y_train[mask]

        # نمونه برداری با جایگذاری فقط روی بخش متغیر
        variable_resampled = resample(variable_subset, n_samples=n_to_generate, replace=True, random_state=42)

        # بخش ثابت هم تکرار می‌کنیم
        fixed_resampled = resample(fixed_subset, n_samples=n_to_generate, replace=True, random_state=42)

        # ستون cluster هم باید به همان مقدار cluster_id تکرار شود
        cluster_resampled = pd.Series([cluster_id]*n_to_generate)

        y_resampled = resample(y_subset, n_samples=n_to_generate, replace=True, random_state=42)

        # ترکیب بخش‌ها
        X_aug_cluster = pd.concat([fixed_resampled.reset_index(drop=True),
                                   variable_resampled.reset_index(drop=True),
                                   cluster_resampled.reset_index(drop=True)], axis=1)
        y_aug_cluster = pd.DataFrame(y_resampled, columns=['CCS'])

        # append به فایل
        X_aug_cluster.to_csv('X_train_augmented.csv', mode='a', header=False, index=False)
        y_aug_cluster.to_csv('y_train_augmented.csv', mode='a', header=False, index=False)

        print(f"داده‌های افزوده‌شده برای کلاستر {cluster_id} ذخیره شد ({n_to_generate} نمونه).")

print("فرآیند افزوده‌سازی داده‌ها تمام شد.")

In [ ]:
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, models

# Load augmented data
X_df = pd.read_csv('X_train_augmented.csv')
y_df = pd.read_csv('y_train_augmented.csv')

# Extract cluster info
clusters = X_df.iloc[:, -1].values  # آخرین ستون = cluster
X_all = X_df.iloc[:, :-1].values    # بقیه ستون‌ها = ویژگی‌ها
y_all = y_df.values.flatten()

unique_clusters = np.unique(clusters)
models_per_cluster = {}

for cluster_id in unique_clusters:
    # داده‌های مربوط به این کلاستر
    X_cluster = X_all[clusters == cluster_id]
    y_cluster = y_all[clusters == cluster_id]
    
    # تعریف مدل
    model = models.Sequential([
        layers.Input(shape=(X_cluster.shape[1],)),
        layers.Dense(1024, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001), loss='mean_absolute_error')

    # آموزش
    print(f"Training model for cluster {cluster_id} with {len(X_cluster)} samples...")
    model.fit(X_cluster, y_cluster, epochs=80, batch_size=100, verbose=0)

    # ذخیره مدل
    model_path = f"model_cluster_{int(cluster_id)}.keras"
    model.save(model_path)
    models_per_cluster[int(cluster_id)] = model_path
